In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()


import mlflow
mlflow.set_experiment("Evaluation_Experiment")
mlflow.set_tracking_uri("http://localhost:5000")

from langchain.chat_models import init_chat_model
llm = init_chat_model("gpt-5-mini-2025-08-07")

/Users/aritrasen/Documents/code/github/mlflow_tutorial/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Create the Agent to evaluate

In [2]:
def create_dummy_agent(question: str) -> str:
    """Function that generates a response for a given query."""
    response = llm.invoke(question)
    return response.content

In [3]:
def qa_predict_function(question: str) -> str:
    """Prediction function for question-answering evaluation."""
    return create_dummy_agent(question)

### Evaluation Dataset

In [4]:
# Define a simple Q&A dataset with questions and expected answers
eval_dataset = [
    {
        "inputs": {"question": "What is the capital of France?"},
        "expectations": {"expected_response": "Paris"},
    },
    {
        "inputs": {"question": "Who was the first person to build an airplane?"},
        "expectations": {"expected_response": "Wright Brothers"},
    },
    {
        "inputs": {"question": "Who wrote Romeo and Juliet?"},
        "expectations": {"expected_response": "William Shakespeare"},
    },
]

### Evaluation criteria using Scorers

Three scorers:

- Correctness: Evaluates if the answer is factually correct, using the "expected_response" field in the dataset.
  
- Guidelines: Evaluates if the answer meets the given guidelines.
- is_concise: A custom scorer defined using the scorer decorator to judge if the answer is concise (less than 5 words).

The first two scorers use LLMs to evaluate the response, so-called LLM-as-a-Judge. 

In [5]:
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines


@scorer
def is_concise(outputs: str) -> bool:
    """Evaluate if the answer is concise (less than 5 words)"""
    return len(outputs.split()) <= 5


scorers = [
    Correctness(),
    Guidelines(name="is_english", guidelines="The answer must be in English"),
    is_concise,
]

The default model used for LLM-as-a-Judge scorers such as Correctness and Guidelines is OpenAI gpt-4o-mini. MLflow supports all major LLM providers, such as Anthropic, Bedrock, Google, xAI, and more through the built-in adapters.

```
# Anthropic
Correctness(model="anthropic:/claude-sonnet-4-20250514")

# Bedrock (Anthropic on Bedrock)
Correctness(model="bedrock:/anthropic.claude-sonnet-4-20250514")

# Bedrock (Amazon Nova)
Correctness(model="bedrock:/amazon.nova-pro-v1:0")
Correctness(model="bedrock:/amazon.nova-lite-v1:0")
Correctness(model="bedrock:/amazon.nova-micro-v1:0")

# Google
Correctness(model="gemini:/gemini-2.5-flash")

# xAI
Correctness(model="xai:/grok-2-latest")
```

In [6]:
results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=qa_predict_function,
        scorers=scorers,
    )

2026/06/12 15:38:55 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/06/12 15:38:55 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 3/3 [Elapsed: 00:22, Remaining: 00:00] [predict_fn: 40%, scorers: 60%]
